# YOLO26 + ByteTrack — independent Colab demonstration
Run in a fresh Google Colab runtime (optionally select a GPU). This notebook is independent of Streamlit and the repository. It creates no virtual environment. Dependencies are installed by uv into a dedicated target directory, not globally. The repository existing `.venv` is never touched. First model use requires internet. Upload a short video; full records use RAM. Counts represent tracker IDs, not guaranteed individuals.

This standalone teaching demo uses direct ByteTrack outputs. The full repository additionally implements temporal confirmation, bounded class voting, image input, and auditable raw observations. Use the repository CLI for the improved pipeline.


In [ ]:
import os, sys, subprocess, pathlib
# Bootstrap the standalone uv tool, then use uv for dependency installation.
subprocess.run(['bash', '-c', 'curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR=/tmp/cv-uv UV_NO_MODIFY_PATH=1 sh'], check=True)
target = '/content/cv-lab-dependencies'
subprocess.run(['/tmp/cv-uv/uv', 'pip', 'install', '--python', sys.executable, '--target', target,
                'ultralytics>=8.4.198,<9', 'lap>=0.5.12', 'imageio-ffmpeg>=0.6', 'pandas>=2.2'], check=True)
sys.path.insert(0, target)
os.environ['PYTHONPATH'] = target + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
import cv2, torch, numpy as np, pandas as pd, time, json, math
from collections import defaultdict, deque, Counter
from ultralytics import YOLO
from google.colab import files
from IPython.display import Video, display
import imageio_ffmpeg
uploaded = files.upload()
if not uploaded:
    raise ValueError('Upload a video first.')
source = next(iter(uploaded))
model = YOLO('yolo26n.pt')
device = 0 if torch.cuda.is_available() else 'cpu'
print('Device:', device)
# ByteTrack initializes on the first track call; a new model resets the sequence.


In [ ]:
capture = cv2.VideoCapture(source)
writer = None
records, frame_times = [], []
unique, trails = defaultdict(set), defaultdict(lambda: deque(maxlen=40))
sides, counted, crossings = {}, set(), Counter()
started = time.perf_counter()
frame_number = 0
try:
    if not capture.isOpened():
        raise ValueError('Cannot decode uploaded video.')
    fps = capture.get(cv2.CAP_PROP_FPS)
    fps = fps if math.isfinite(fps) and fps > 0 else 30.0
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        frame_number += 1
        height, width = frame.shape[:2]
        scale = min(1., 1280 / width)
        width, height = max(2, int(width*scale)//2*2), max(2, int(height*scale)//2*2)
        frame = cv2.resize(frame, (width, height))
        if writer is None:
            writer = cv2.VideoWriter('annotated_mpeg4.mp4', cv2.VideoWriter_fourcc(*'mp4v'), fps, (width,height))
            if not writer.isOpened():
                raise RuntimeError('Video writer unavailable.')
        t = time.perf_counter()
        result = model.track(frame, persist=True, tracker='bytetrack.yaml', conf=.1,
                             iou=.5, imgsz=640, device=device, nms=None, verbose=False)[0]
        frame_times.append(time.perf_counter()-t)
        canvas = result.plot()
        boxes = result.boxes
        if boxes is not None and boxes.id is not None:
            for box, cls, score, tid in zip(boxes.xyxy.cpu().numpy(), boxes.cls.cpu().numpy(),
                                           boxes.conf.cpu().numpy(), boxes.id.cpu().numpy()):
                tid, label = int(tid), result.names[int(cls)]
                x1,y1,x2,y2 = map(float,box)
                cx,cy = (x1+x2)/2,(y1+y2)/2
                unique[label].add(tid)
                trails[tid].append((int(cx),int(cy)))
                color = (80+(tid*37)%176,80+(tid*71)%176,80+(tid*109)%176)
                if len(trails[tid])>1:
                    cv2.polylines(canvas,[np.array(trails[tid],np.int32)],False,color,2)
                # Finite horizontal line, 8-pixel dead band, one crossing/ID/direction.
                distance = cy-height/2
                if abs(distance)>8:
                    side = 1 if distance>0 else -1
                    old = sides.get(tid)
                    if old and old[0]!=side:
                        old_x,old_y = old[1]
                        fraction = (height/2-old_y)/(cy-old_y)
                        cross_x = old_x+fraction*(cx-old_x)
                        direction = 'A to B' if side>0 else 'B to A'
                        if .1*width<=cross_x<=.9*width and (tid,direction) not in counted:
                            crossings[(label,direction)]+=1
                            counted.add((tid,direction))
                    sides[tid]=(side,(cx,cy))
                records.append([tid,label,float(score),x1,y1,x2,y2,cx,cy,frame_number,(frame_number-1)/fps])
        cv2.line(canvas,(int(.1*width),height//2),(int(.9*width),height//2),(0,255,255),2)
        writer.write(canvas)
finally:
    capture.release()
    if writer is not None:
        writer.release()
if frame_number == 0:
    raise ValueError('No decoded frames.')
elapsed = time.perf_counter()-started
print(f'Processed {frame_number} frames in {elapsed:.2f}s ({frame_number/elapsed:.2f} FPS)')

In [ ]:
columns = ['track_id','object_class','confidence','x1','y1','x2','y2','centroid_x','centroid_y','frame_number','timestamp']
pd.DataFrame(records, columns=columns).to_csv('tracking.csv',index=False)
counts = [{'object_class':label,'unique_objects':len(ids),'a_to_b':crossings[(label,'A to B')],
           'b_to_a':crossings[(label,'B to A')]} for label,ids in unique.items()]
pd.DataFrame(counts,columns=['object_class','unique_objects','a_to_b','b_to_a']).to_csv('counts.csv',index=False)
summary = {'frames_processed':frame_number,'processing_seconds':elapsed,'processing_fps':frame_number/elapsed,
           'average_inference_ms':1000*sum(frame_times)/len(frame_times),'counts':counts}
pathlib.Path('analytics.json').write_text(json.dumps(summary,indent=2))
output = 'annotated.mp4'
try:
    subprocess.run([imageio_ffmpeg.get_ffmpeg_exe(),'-y','-i','annotated_mpeg4.mp4','-c:v','libx264',
                    '-pix_fmt','yuv420p','-movflags','+faststart',output],check=True,capture_output=True)
    display(Video(output,embed=True))
except (OSError,RuntimeError,subprocess.SubprocessError):
    output='annotated_mpeg4.mp4'
    print('H.264 unavailable; download the MPEG-4 fallback for external playback.')
display(pd.DataFrame(counts))

In [ ]:
import zipfile
with zipfile.ZipFile('tracking_results.zip','w',zipfile.ZIP_DEFLATED) as archive:
    for name in [output,'tracking.csv','counts.csv','analytics.json']:
        archive.write(name)
files.download('tracking_results.zip')

## Interpretation
The line runs left to right: A is above and B below. Each ID counts once in each direction. Class changes and ID switches can inflate unique counts. Confidence is not accuracy; no mAP/MOTA/IDF1 is reported. Full-video timing excludes final encoding/export. The notebook records tracked detections only; the repository also preserves untracked detection records. Source audio is omitted. To run a new independent video, rerun model initialization before processing.